# NumCompute Quickstart Demo

This notebook demonstrates an end-to-end workflow using the NumCompute toolkit, including preprocessing, statistics, ranking, metrics, and benchmarking.

In [67]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import time

from numcompute.io import read_csv
from numcompute.preprocessing import StandardScaler, MinMaxScaler, Imputer, OneHotEncoder
from numcompute.stats import mean, var, quantile, histogram, Welford
from numcompute.sort_search import stable_sort, multi_key_sort, topk, quickselect, binary_search
from numcompute.rank import rank, percentile
from numcompute.metrics import accuracy, precision, recall, f1, confusion_matrix, mse
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline

In [68]:
csv_path = "sample_demo.csv"

with open(csv_path, "w", encoding="utf-8") as f:
    f.write("feature1,feature2\n")
    f.write("1,10\n")
    f.write("2,\n")
    f.write("3,30\n")
    f.write("4,40\n")

X = read_csv(csv_path, skip_header=True)

print("CSV data:")
print(X)

CSV data:
[[1.0 10.0]
 [2.0 nan]
 [3.0 30.0]
 [4.0 40.0]]


In [69]:
X = X.astype(float)

imputer = Imputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

minmax = MinMaxScaler()
X_minmax = minmax.fit_transform(X_imputed)

print("After imputation:")
print(X_imputed)

print("\nAfter StandardScaler:")
print(X_scaled)

print("\nAfter MinMaxScaler:")
print(X_minmax)

After imputation:
[[ 1.         10.        ]
 [ 2.         26.66666667]
 [ 3.         30.        ]
 [ 4.         40.        ]]

After StandardScaler:
[[-1.34164079 -1.5430335 ]
 [-0.4472136   0.        ]
 [ 0.4472136   0.3086067 ]
 [ 1.34164079  1.2344268 ]]

After MinMaxScaler:
[[0.         0.        ]
 [0.33333333 0.55555556]
 [0.66666667 0.66666667]
 [1.         1.        ]]


In [70]:
categories = np.array(["red", "blue", "red", "green"])

encoder = OneHotEncoder()
encoded = encoder.fit_transform(categories)

print("Original categories:")
print(categories)

print("\nOne-hot encoded:")
print(encoded)

Original categories:
['red' 'blue' 'red' 'green']

One-hot encoded:
[[0 0 1]
 [1 0 0]
 [0 0 1]
 [0 1 0]]


In [71]:
print("Mean:", mean(X_imputed, axis=0))
print("Variance:", var(X_imputed, axis=0))
print("Median:", quantile(X_imputed, 0.5))

hist, edges = histogram(X_imputed[:, 0], bins=3)
print("Histogram:", hist)
print("Bin edges:", edges)

wf = Welford()
wf.update(X_imputed[:, 0])
print("Welford result:", wf.finalize())

Mean: [ 2.5        26.66666667]
Variance: [  1.25       116.66666667]
Median: 7.0
Histogram: [1 1 2]
Bin edges: [1. 2. 3. 4.]
Welford result: {'count': 4, 'mean': np.float64(2.5), 'variance': np.float64(1.25)}


In [72]:
arr = np.array([5, 1, 9, 3, 7])

print("Original array:", arr)
print("Stable sort:", stable_sort(arr))
print("Stable sort descending:", stable_sort(arr, ascending=False))

values, indices = topk(arr, 3)
print("Top 3 values:", values)
print("Top 3 indices:", indices)

print("Quickselect 3rd smallest:", quickselect(arr, 2))

sorted_arr = np.array([1, 3, 5, 7, 9])
idx, found = binary_search(sorted_arr, 5)

print("Binary search index:", idx)
print("Found:", found)

Original array: [5 1 9 3 7]
Stable sort: [1 3 5 7 9]
Stable sort descending: [9 7 5 3 1]
Top 3 values: [9 7 5]
Top 3 indices: [2 4 0]
Quickselect 3rd smallest: 5
Binary search index: 2
Found: True


In [73]:
data = np.array([
    [2, 30],
    [1, 20],
    [2, 10],
])

sorted_data = multi_key_sort(data, keys=[0, 1], ascending=[True, True])

print("Original data:")
print(data)

print("\nMulti-key sorted data:")
print(sorted_data)

Original data:
[[ 2 30]
 [ 1 20]
 [ 2 10]]

Multi-key sorted data:
[[ 1 20]
 [ 2 10]
 [ 2 30]]


In [74]:
rank_data = np.array([100, 50, 50, 25])

print("Data:", rank_data)
print("Ordinal rank:", rank(rank_data, method="ordinal"))
print("Dense rank:", rank(rank_data, method="dense"))
print("Average rank:", rank(rank_data, method="average"))

percentile_data = np.array([1, 2, 3, 4, 5])
print("50th percentile:", percentile(percentile_data, 50))
print("0, 50, 100 percentiles:", percentile(percentile_data, [0, 50, 100]))

Data: [100  50  50  25]
Ordinal rank: [4. 2. 3. 1.]
Dense rank: [3. 2. 2. 1.]
Average rank: [4.  2.5 2.5 1. ]
50th percentile: 3.0
0, 50, 100 percentiles: [1. 3. 5.]


In [75]:
y_true = np.array([1, 0, 1, 1])
y_pred = np.array([1, 0, 0, 1])

print("Accuracy:", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall:", recall(y_true, y_pred))
print("F1:", f1(y_true, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

y1 = np.array([1, 2, 3])
y2 = np.array([1, 2, 4])

print("MSE:", mse(y1, y2))

Accuracy: 0.75
Precision: 1.0
Recall: 0.6666666666666666
F1: 0.8
Confusion matrix:
[[1 0]
 [1 2]]
MSE: 0.3333333333333333


In [76]:
def f(x):
    return x[0] ** 2 + x[1] ** 2

x = np.array([3.0, 4.0])
print("Gradient:", grad(f, x, method="central"))


def F(x):
    return np.array([
        x[0] + x[1],
        x[0] * x[1]
    ])

x2 = np.array([2.0, 3.0])
print("Jacobian:")
print(jacobian(F, x2))

Gradient: [6. 8.]
Jacobian:
[[1. 1.]
 [3. 2.]]


In [77]:
X_pipe = np.array([
    [1.0],
    [2.0],
    [3.0]
])

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("minmax", MinMaxScaler())
])

X_pipe_out = pipe.fit_transform(X_pipe)

print("Original pipeline data:")
print(X_pipe)

print("\nPipeline output:")
print(X_pipe_out)

Original pipeline data:
[[1.]
 [2.]
 [3.]]

Pipeline output:
[[0. ]
 [0.5]
 [1. ]]


In [78]:
a = np.random.rand(200000)
b = np.random.rand(200000)

start = time.time()
loop_result = [a[i] + b[i] for i in range(len(a))]
loop_time = time.time() - start

start = time.time()
vectorized_result = a + b
vectorized_time = time.time() - start

print("Loop time:", loop_time)
print("Vectorized time:", vectorized_time)
print("Vectorized faster:", vectorized_time < loop_time)

Loop time: 0.029094934463500977
Vectorized time: 0.0004899501800537109
Vectorized faster: True
